# Can we fit $\mathcal{H}$ Using DNNs at a Single Kinematic Point Using the KM15/BKM10 Formalism for the Cross-Section **using Experimental Data Only**?

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Native Libraries:

In [ ]:
import os
import datetime

### (1.2): Import 3rd-Party Libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import corner
import seaborn as sns

import gepard as g
from gepard.fits import th_KM15

from bkm10_lib.core import DifferentialCrossSection
from bkm10_lib.inputs import BKM10Inputs
from bkm10_lib.cff_inputs import CFFInputs

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")
print(f"[INFO]: corner version: {corner.__version__}")
print(f"[INFO]: seaborn version: {sns.__version__}")

### (1.4): Customizing Plotting Settings:

In [ ]:
plt.rcParams.update({"text.usetex": True, "font.family": "serif"})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 3.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 3.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['savefig.dpi'] = 300

## (2): Data Formatting/Collection Settings:

In [ ]:
VERSION_NUMBER = 3
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

## (3): Data Loading and Analysis:

### (3.1): Loading in `gepard`'s Datasets:

**Remark:** `g.dset` is a dictionary of `gepard` `DataSet` classes.
```python
g.dset[73]
> DataSet with 39 points
```

**Remark:** Some of the `DataPoint`s in the `DataSet`s *do not have all the right kinematics!* See below for this:

In [ ]:
try:
    print(f"[INFO]: k = {g.dset[47][0].in1energy}")
    print(f"[INFO]: xb = {g.dset[47][0].xB}")
    print(f"[INFO]: t = {g.dset[47][0].t}")
    print(f"[INFO]: Q^2 = {g.dset[47][0].Q2}")
except AttributeError:
    print("[ERROR]: Missing crucial kinematic setting information!")

### (3.2): Determining which `gepard` `DataSet`s have all of the right Kinematic Variables:

[NOTE]: We are looking for $k$, $x_{\textrm{B}}$, $t$, and $Q^{2}$.

In [ ]:
required_attributes = ['xB', 't', 'Q2', 'in1energy', 'observable', 'val', 'err']
valid_datasets = []

for dataset_index, dataset in g.dset.items():

    # check the first datapoint in the DataSet for contents!
    first_gepard_datapoint = dataset[0] if len(dataset) > 0 else None
    
    if first_gepard_datapoint and all(hasattr(first_gepard_datapoint, kinematic_attribute) for kinematic_attribute in required_attributes):
        valid_datasets.append(dataset_index)

print(f"[INFO]: Valid dataset indices are:\n{sorted(valid_datasets)}")
print(f"[INFO]: Length of valid datasets = {len(valid_datasets)}")
print(f"[INFO]: Compare length of *all* dataset = {len(g.dset)}")
print(f"[INFO]: Total invalid (according to our criteria) datasets = {len(g.dset) - len(valid_datasets)}")

According to [gepard documentation](https://gepard.phy.hr/docs/observables.htmlhttps://gepard.phy.hr/docs/observables.html), here are the relevant observables for our analysis

1. `XGAMMA` = basically DVCS cross-section
2. `XUU` = beam-averaged cross-section
1. `AC` = beam charge asymmetry
1. `ALU` = beam spin asymmetry
2. `AUL` = longitudinally-polarized target asymmetry
3. `AUT` = transversely-polarized target asymmetry

We need to figure out which `DataSet`s have the observables we're looking for!

### (3.3): Paritioning Datasets into Dictionary of Desired Observables

In [ ]:
# target_observables = { 'XUU', 'ALU', 'AC', 'AUL', 'TSA', 'XGAMMA' }
target_observables = { 'XUU', 'ALU' }
unknown_observables = {}

# dictionary of string-to-list key-value pairs:
desired_observable_dictionary = { observable: [] for observable in target_observables }

# required attributes of `DataPoint`
required_attributes = ['in1energy', 'xB', 't', 'Q2',  'observable', 'val', 'err']

valid_datasets = []
invalid_datasets = {}

for dataset_index, dataset in g.dset.items():

    if not dataset:
        invalid_datasets[dataset_index] = "Dataset is empty"
        continue

    missing_attributes = []

    for point_index, datapoint in enumerate(dataset):

        missing = [ attribute for attribute in required_attributes if not hasattr(datapoint, attribute) ]

        if missing:
            missing_attributes.append({ "point_index": point_index, "missing": missing, })

    if missing_attributes:
        invalid_datasets[dataset_index] = {
            "reason": "Missing required attributes",
            "details": missing_attributes,
        }
        continue

    observables_found = { datapoint.observable for datapoint in dataset }

    if len(observables_found) != 1:
        invalid_datasets[dataset_index] = {
            "reason": "Mixed observables detected",
            "observables": sorted(observables_found),
        }
        continue

    observable_name = next(iter(observables_found))
    observable_name = observable_name.strip().upper()

    valid_datasets.append(dataset_index)

    if observable_name in desired_observable_dictionary:
        desired_observable_dictionary[observable_name].append(dataset_index)
    else:
        if observable_name not in unknown_observables:
            unknown_observables[observable_name] = []
        unknown_observables[observable_name].append(dataset_index)

print(f"[INFO]: Total number of datasets in gepard: {len(g.dset)}")
print(f"[INFO]: Length of valid datasets = {len(valid_datasets)}")
print(f"[INFO]: Length of invalid datasets = {len(invalid_datasets)}")
print(f"[INFO]: Valid dataset indices are:\n{sorted(valid_datasets)}")
print(f"[INFO]: Invalid dataset indices are:\n{sorted(invalid_datasets)}")
print(f"[INFO]: Does this metric match? It should: {len(g.dset) - len(valid_datasets) == len(invalid_datasets)}")
assert len(g.dset) - len(valid_datasets) == len(invalid_datasets), "[ASSERT]: Unexpected length of invalid datasets..."

for observable_name in target_observables:
    indices = desired_observable_dictionary[observable_name]
    print(f"[INFO]: {observable_name} has set indices of: {sorted(indices)}")

[NOTE]: From the brief code snippet above, we see that the *majority* of `gepard` `DataSet`s involve `"XUU"` and `"ALU"`, which justifies a simultaneous fit in the observables $d^{4}\sigma^{UU}$ and $\textrm{BSA}\left( 0 \right)$.

So, the following print should match the numbers above:

In [ ]:
for observable_key, experiment_ids in desired_observable_dictionary.items():
    print(observable_key, sorted(experiment_ids))

### (3.4): What are the other observables?

In [ ]:
for observable_name, indices in sorted(unknown_observables.items()):
    print(f"{observable_name}: {indices}")

### (3.5): Make **massive datafile** of experimental data:

In [ ]:
# rows for model predictions:
rows_for_experiment_w_ground_truth = []
# rows for raw experimental data:
rows_for_experimental_data_only = []

total_datapoints = 0
total_datapoints_without_observables = 0
total_unrecognized_observables = 0
total_datapoints_missing_kinematics = 0

for observable_key, experiment_ids in desired_observable_dictionary.items():
    for experiment_id in sorted(experiment_ids):

        # query the dataset from gepard:
        dataset = g.dset[experiment_id]
        print(f"[INFO]: Experiment {dataset.collaboration} ({dataset.year}), ID = {experiment_id}, {len(dataset)} datapoints")
        
        for datapoint_index, datapoint in enumerate(dataset):

            total_datapoints = total_datapoints + 1

            # this should always pass:
            if not hasattr(datapoint, "observable"):
                print(f"[WARN]: Datapoint for Experiment {dataset.collaboration} ({dataset.year}) ID = {experiment_id} has no observable...")
                total_datapoints_without_observables += 1
            
            # check if the datapoint has all of the required kinematic variables...
            if all(hasattr(datapoint, attr) for attr in ["in1energy", "xB", "Q2", "t", "phi"]):
            
                # predict KM15 CFFs using Gepard's KM15:
                km15_real_h = th_KM15.ReH(datapoint)
                km15_imag_h = th_KM15.ImH(datapoint)
                km15_real_e = th_KM15.ReE(datapoint)
                km15_imag_e = th_KM15.ImE(datapoint)
                km15_real_ht = th_KM15.ReHt(datapoint)
                km15_imag_ht = th_KM15.ImHt(datapoint)
                km15_real_et = th_KM15.ReEt(datapoint)
                km15_imag_et = th_KM15.ImEt(datapoint)

                # initialize a BKM10 computation hub:
                km15_bkm10_cross_section = DifferentialCrossSection(
                    configuration = {
                        "kinematics": BKM10Inputs(
                            lab_kinematics_k = datapoint.in1energy,
                            squared_Q_momentum_transfer = datapoint.Q2,
                            x_Bjorken = datapoint.xB,
                            squared_hadronic_momentum_transfer_t = datapoint.t),
                        "cff_inputs": CFFInputs(
                            compton_form_factor_h = complex(km15_real_h, km15_imag_h),
                            compton_form_factor_h_tilde = complex(km15_real_ht, km15_imag_ht),
                            compton_form_factor_e = complex(km15_real_e, km15_imag_e),
                            compton_form_factor_e_tilde = complex(km15_real_et, km15_imag_et)),
                        "using_ww": True
                    },
                    verbose = False, debugging = False)
            
                # compute cross-section (XUU)
                unpolarized_cross_section = km15_bkm10_cross_section.compute_cross_section(
                    datapoint.phi, lepton_helicity = 0.0, target_polarization = 0.0).real
                # compute BSA (ALU)
                bkm10_bsa_km15 = km15_bkm10_cross_section.compute_bsa(
                    datapoint.phi, target_polarization = 0.0).real

                # need to initialize these garbage variables for looping purposes:
                exp_xsec, exp_xsec_err, exp_xsec_errstat, exp_xsec_errsyst = 0.0, 0.0, 0.0, 0.0 # beam-averaged cross-section
                exp_bsa, exp_bsa_err, exp_bsa_errstat, exp_bsa_errsyst = 0.0, 0.0, 0.0, 0.0 # beam-spin asym.
                    
                if observable_key == 'XUU': # UNPOLARIZED CROSS SECTION
                    exp_xsec = datapoint.val
                    exp_xsec_err = datapoint.err
                    exp_xsec_errstat = getattr(datapoint, "errstat", datapoint.err)
                    exp_xsec_errsyst = getattr(datapoint, "errsyst", datapoint.err)

                elif observable_key == 'ALU': # BSA
                    exp_bsa = datapoint.val
                    exp_bsa_err = datapoint.err
                    exp_bsa_errstat = getattr(datapoint, "errstat", datapoint.err)
                    exp_bsa_errsyst = getattr(datapoint, "errsyst", datapoint.err)

                else:
                    print(f"[ERROR]: Unrecognized observable key: {observable_key}")
                    total_unrecognized_observables += 1

                # this is the *row* we will insert into the dataframe:
                experimental_data_point = {
                    "experiment_id": experiment_id,
                    "k": datapoint.in1energy, "q_squared": datapoint.Q2,
                    "x_b": datapoint.xB, "t": datapoint.t,
                    "phi": datapoint.phi,
                    "unp_beam_unp_target_xsec": exp_xsec,
                    "unp_beam_unp_target_xsec_err": exp_xsec_err,
                    "unp_beam_unp_target_xsec_errstat": exp_xsec_errstat,
                    "unp_beam_unp_target_xsec_errsyst": exp_xsec_errsyst,
                    "unp_target_bsa": exp_bsa,
                    "unp_target_bsa_err": exp_bsa_err,
                    "unp_target_bsa_errstat": exp_bsa_errstat,
                    "unp_target_bsa_errsyst": exp_bsa_errsyst,
                    "Re[H]": km15_real_h, "Im[H]": km15_imag_h,
                    "Re[E]": km15_real_e, "Im[E]": km15_imag_e,
                    "Re[Ht]": km15_real_ht, "Im[Ht]": km15_imag_ht,
                    "Re[Et]": km15_real_et, "Im[Et]": km15_imag_et,
                    "coordinate_frame": datapoint.frame,
                    "experiment_year": f"{dataset.collaboration}_{dataset.year}",
                    "flag": "unknown"
                }

                pseudodata_point = experimental_data_point.copy()
                pseudodata_point.update({
                    "unp_beam_unp_target_xsec": unpolarized_cross_section[0], # [0] index needed because datapoint.phi is *not* an array...
                    "unp_target_bsa": bkm10_bsa_km15[0], # [0] index needed because datapoint.phi is *not* an array...
                })

                rows_for_experiment_w_ground_truth.append(pseudodata_point)
                rows_for_experimental_data_only.append(experimental_data_point)

                del pseudodata_point
                del experimental_data_point
                del km15_bkm10_cross_section

            else:
                print(f"[WARN]: Missing kinematics for datapoint {datapoint_index + 1} in {dataset.collaboration}, ID = {experiment_id}")
                total_datapoints_missing_kinematics += 1

In [ ]:
print(f"[INFO]: Total datapoints iterated over: {total_datapoints}")
print(f"[INFO]: Total datapoints without appropriate kinematics: {total_datapoints_missing_kinematics}")
print(f"[INFO]: Total datapoints without observable property: {total_datapoints_without_observables}: ")
print(f"[INFO]: Total datapoints with an unrecognized observable: {total_unrecognized_observables}")
print(f"[INFO]: The following should evaluate to True: {total_datapoints - total_datapoints_missing_kinematics + total_datapoints_without_observables + total_unrecognized_observables == len(rows_for_experiment_w_ground_truth)}")

print(f"[INFO]: Total valid rows added: {len(rows_for_experiment_w_ground_truth)}")

### (3.6): Actually **save** the raw datafile --- *with the "set" column!*

In [ ]:
# these columns define a *kinematic setting*:
kinematic_columns = ['k', 'x_b', 'q_squared', 't']

##########################################################################################
# [NOTE]: this part of the code makes a dataset with "ground truth" values:
##########################################################################################

df_exp_data_w_ground_truth = pd.DataFrame(rows_for_experiment_w_ground_truth)
print(f"[INFO]: Total number of rows in exp-derived DF: {len(df_exp_data_w_ground_truth)}")

# this relabels the sets actually...
df_exp_data_w_ground_truth['set'] = df_exp_data_w_ground_truth.groupby(kinematic_columns, sort = False).ngroup() + 1
unique_sets_exp_derived = df_exp_data_w_ground_truth['set'].nunique()
print(f"[INFO]: Total unique kinematic settings in the exp-derived DF: {unique_sets_exp_derived}")

df_exp_data_w_ground_truth.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/main_pseudodata_file_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)

##########################################################################################
# [NOTE]: this part of the code makes a dataset with *only* the experimental data---not any CFF info!
##########################################################################################

df_experimental_data = pd.DataFrame(rows_for_experimental_data_only)
print(f"[INFO]: Total number of rows in exp-only DF: {len(df_experimental_data)}")

# relabeling the sets:
df_experimental_data['set'] = df_experimental_data.groupby(kinematic_columns, sort = False).ngroup() + 1
unique_sets_exp_data = df_experimental_data['set'].nunique()
print(f"[INFO]: Total unique kinematic settings in the  exp-only DF: {unique_sets_exp_data}")

df_experimental_data.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/experimental_data_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)

assert len(df_experimental_data) == len(df_exp_data_w_ground_truth),"[ASSERT]: Mismatch in number of expected rows from both datasets."

### (3.7): Scrub the dataset for redundancies in data points:

#### (3.7.1): Initial dropping of duplicate rows:

In [ ]:
print(f"[INFO]: Number of initial rows: {len(df_experimental_data)}")
df_cleaned = df_experimental_data.drop_duplicates().copy()
print(f"[INFO]: Number of remaining rows: {len(df_cleaned)}")

#### (3.7.2): Define the observable columns:

In [ ]:
# [NOTE]: Remember that we're not doing BCA, TSA, or XGAMMA yet:
# observable_columnnames = [
#     "unp_beam_unp_target_xsec", "unp_beam_unp_target_xsec_err", "unp_beam_unp_target_xsec_errsyst", "unp_beam_unp_target_xsec_errstat",
#     "unp_target_bsa", "unp_target_bsa_err", "unp_target_bsa_errsyst", "unp_target_bsa_errstat", 
#     "unp_target_bca", "unp_target_bca_err", "unp_target_bca_errsyst", "unp_target_bca_errstat", 
#     "unp_target_lp_target_xsec", "unp_target_lp_target_xsec_err", "unp_target_lp_target_xsec_errsyst", "unp_target_lp_target_xsec_errstat", 
#     "unp_target_tp_target_xsec", "unp_target_tp_target_xsec_err", "unp_target_tp_target_xsec_errsyst", "unp_target_tp_target_xsec_errstat", 
#     "unp_target_xgamma", "unp_target_xgamma_err", "unp_target_xgamma_errsyst", "unp_target_xgamma_errstat", 
#     ]

observable_columnnames = [
    "unp_beam_unp_target_xsec", "unp_beam_unp_target_xsec_err", "unp_beam_unp_target_xsec_errsyst", "unp_beam_unp_target_xsec_errstat",
    "unp_target_bsa", "unp_target_bsa_err", "unp_target_bsa_errsyst", "unp_target_bsa_errstat", 
    ]

cross_section_columnnames = ["unp_beam_unp_target_xsec", "unp_beam_unp_target_xsec_err", "unp_beam_unp_target_xsec_errsyst", "unp_beam_unp_target_xsec_errstat"]
bsa_column_names = [ "unp_target_bsa", "unp_target_bsa_err", "unp_target_bsa_errsyst", "unp_target_bsa_errstat" ]

#### (3.7.3): Group the dataframe by these unique values:

In [ ]:
# is groupby stochastic?
df_unique_experimental_settings = df_cleaned.groupby(
    ['k', 'q_squared', 'x_b', 't', 'phi'],
    sort = False)

print(f"[INFO]: Found {len(df_unique_experimental_settings)} unique kinematic points.")

### (3.8): Construction of the Dataset containing **only nonzero $d^{4}\sigma^{UU}$ rows!**

In [ ]:
final_cross_section_rows = []

number_of_zero_cross_sections = 0
number_of_nonzero_cross_sections = 0

# https://stackoverflow.com/a/29262040 -> for how to iterate across df rows:
for row_index, row in df_cleaned.iterrows():
    # [NOTE]: this is a *proxy* for the errorbars on the cross-section too!
    if row["unp_beam_unp_target_xsec"] != 0.0:
        number_of_nonzero_cross_sections += 1
        final_cross_section_rows.append(row)
    else:
        number_of_zero_cross_sections += 1

print(f"[INFO]: zero cross-sections: {number_of_zero_cross_sections}, nonzero cross-sections: {number_of_nonzero_cross_sections}")
print(f"[INFO]: The sum of the two is: {number_of_zero_cross_sections + number_of_nonzero_cross_sections}")
print(f"[INFO]: This should be true: {number_of_zero_cross_sections + number_of_nonzero_cross_sections == len(df_cleaned)}")
print(f"[INFO]: This should also be true: {len(final_cross_section_rows) == number_of_nonzero_cross_sections}")

df_cross_section_no_zeros = pd.DataFrame(final_cross_section_rows)

df_cross_section_no_zeros.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/refined_cross_section_data_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)  

### (3.9): We now scrub the **cross-section** dataset for two things: **gigantic error bars** that imply **nonphysical cross-sections**:

In [ ]:
bad_sets = set()

for row_index, row in df_cross_section_no_zeros.iterrows():
    row_cross_section_value = row["unp_beam_unp_target_xsec"]
    
    # [NOTE]: the 2 is two sigma:
    row_cross_section_lower_value = row_cross_section_value - 2.0 * row["unp_beam_unp_target_xsec_err"]

    # [NOTE]: this number is magic, but it's kind of a good estimate:
    # it says "if the 2sigma bar is below -0.2, then we throw out this
    # point." Typically, the insane error bars give cross-sections of
    # -9.0 and -15.0, so this lower bound is acceptable...
    if row_cross_section_lower_value < -0.2:
        bad_sets.add(row["set"])

print(f"[INFO]: Sets to remove: {sorted(bad_sets)}")

# quotient out these invalid settings:
# https://stackoverflow.com/a/18173074 -> how to do basic Pandas dataframe conditioning:
df_cross_section_no_huge_errors = df_cross_section_no_zeros[~df_cross_section_no_zeros["set"].isin(bad_sets)]

del df_cross_section_no_zeros

### (3.10): We now scrub the cross-section dataframe for kinematic sets with **sparse data** (i.e. only **two datapoints**):

[NOTE]: Some lore about (what you will find) is the single bad set in the dataset. It corresponds to kinematic set number `413`, and it actually had many `DataPoint`s in it already. Unfortunately, due to the previous scrubbing procedure---the one where we erased rows with $0$ cross-section values---*two non-zero values* of the cross-section were retained! If you look in `experimental_data_vX_Y.csv`, it should be rows `4380` and `4381`; you will discover all the "surrounding rows" within the same kinematic setting have $0.0$ as their cross-section value.

In [ ]:
cross_section_set_count = df_cross_section_no_huge_errors.groupby("set").size()
print(f"[INFO]: Total unique settings in the cross-section dataframe: {len(cross_section_set_count)}")
cross_section_single_phi_sets = cross_section_set_count[cross_section_set_count == 1].index

print(f"[INFO]: Found {len(cross_section_single_phi_sets)} sets with only one phi value.")
print(sorted(cross_section_single_phi_sets))

cross_section_double_phi_sets = cross_section_set_count[cross_section_set_count == 2].index

print(f"[INFO]: Found {len(cross_section_double_phi_sets)} sets with only two phi values.")
print(sorted(cross_section_double_phi_sets))

print(f"[INFO]: Total number of rows in the final dataset: {len(df_cross_section_no_huge_errors)}")

### (3.11): Now, we actually *remove* this bad set from the dataset:

In [ ]:
cross_section_single_phi_sets = set(cross_section_single_phi_sets)
cross_section_double_phi_sets = set(cross_section_double_phi_sets)

print(f"[INFO]: Sets to remove for only one phi value: {sorted(cross_section_single_phi_sets)}")
print(f"[INFO]: Sets to remove for only two phi values: {sorted(cross_section_double_phi_sets)}")

df_cross_section_no_issues = df_cross_section_no_huge_errors[~df_cross_section_no_huge_errors["set"].isin(cross_section_single_phi_sets)].copy()
print(f"[INFO]: Current number of rows in dataset: {len(df_cross_section_no_huge_errors)}")

df_cross_section_no_issues = df_cross_section_no_huge_errors[~df_cross_section_no_huge_errors["set"].isin(cross_section_double_phi_sets)].copy()
print(f"[INFO]: Total number of rows in final dataset: {len(df_cross_section_no_issues)}")

df_cross_section_no_issues.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/refined_cross_section_data_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)

### (3.12): Construction of the Dataset containing **only nonzero $\text{BSA}$ rows!**

In [ ]:
final_bsa_rows = []

number_of_zero_bsa = 0
number_of_nonzero_bsa = 0

# https://stackoverflow.com/a/29262040 -> for how to iterate across df rows:
for row_index, row in df_cleaned.iterrows():
    # [NOTE]: this is a *proxy* for the errorbars on the BSA!
    if row["unp_target_bsa"] != 0.0:
        number_of_nonzero_bsa += 1
        final_bsa_rows.append(row)
    else:
        number_of_zero_bsa += 1

print(f"[INFO]: zero BSAs: {number_of_zero_bsa}, nonzero BSAs: {number_of_nonzero_bsa}")
print(f"[INFO]: The sum of the two is: {number_of_zero_bsa + number_of_nonzero_bsa}")
print(f"[INFO]: This should be true: {number_of_zero_bsa + number_of_nonzero_bsa == len(df_cleaned)}")
print(f"[INFO]: This should also be true: {len(final_bsa_rows) == number_of_nonzero_bsa}")

df_bsa_no_zeros = pd.DataFrame(final_bsa_rows)

df_bsa_no_zeros.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/refined_bsa_data_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)

In [ ]:
# [NOTE]: the magic number 9550 is the one we get if we Ctrl+F 0.0,0.0,0.0,0.0, because we assume
# that if the observable value is 0.0 then so is its three experimental uncertainties. So, the total
# number that appears in the Ctrl+F search bar will be *both* the number of times the cross-section
# is 0.0 and the BSA is 0.0.
print(f"[INFO]: Quick check: {number_of_zero_cross_sections + number_of_zero_bsa == 9550}")

### (3.13): We now scrub the **BSA** dataset for settings with *only a single datapoint!*

This is an annoying situation, but we have to do it.

##### (3.13.1): If a given `set` has only a single row, it indicates there exists only a single datapoint:

[NOTE]: This might not work if there are duplicate rows with the same data. Damn... In other words, there might be two rows with exactly the same data, and the plot of the BSA vs. $\phi$ would not indicate any problems visually.

In [ ]:
set_counts = df_bsa_no_zeros.groupby("set").size()
print(f"[INFO]: Total unique settings in the BSA dataframe: {len(set_counts)}")
single_phi_sets = set_counts[set_counts == 1].index

print(f"[INFO]: Found {len(single_phi_sets)} sets with only one phi value.")
print(sorted(single_phi_sets))

# [NOTE]: it turned out that there were kinematic settings with only TWO phi points...
# So, we need to run this thing again:
double_phi_sets = set_counts[set_counts == 2].index

print(f"[INFO]: Found {len(double_phi_sets)} sets with only two phi values.")
# numbers should be: 138, 145, 174, 214, 226, 255, 321 (I looked at version_2_2's plots!)
print(sorted(double_phi_sets))

##### (3.13.2): Actually cutting out the bad sets:

In [ ]:
single_phi_bad_sets = set(single_phi_sets)
double_phi_bad_sets = set(double_phi_sets)

print(f"[INFO]: Sets to remove: {sorted(single_phi_bad_sets)}")
print(f"[INFO]: Sets to remove: {sorted(double_phi_bad_sets)}")

df_bsa_no_issues = df_bsa_no_zeros[~df_bsa_no_zeros["set"].isin(single_phi_bad_sets)].copy()
print(f"[INFO]: Current number of rows in dataset: {len(df_bsa_no_issues)}")

df_bsa_no_issues = df_bsa_no_issues[~df_bsa_no_issues["set"].isin(double_phi_bad_sets)].copy()
print(f"[INFO]: Total number of rows in final dataset: {len(df_bsa_no_issues)}")

df_bsa_no_issues.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/refined_bsa_data_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)

### (3.14): Construction of the *combined* Dataset:

[NOTE]: We are still working on this...

In [ ]:
merged_rows = []

number_of_fragmented_points = 0
number_of_successful_merges = 0
number_of_all_zero_observables = 0
number_of_fully_zero_rows_removed = 0
number_of_conflicts = 0

for group_key, group_df in df_unique_experimental_settings:

    # if there is a *single row* in this group:
    if len(group_df) == 1:
        # just take the entire row and shove it into the list:
        merged_rows.append(group_df.iloc[0])
        continue

    # print(f"[INFO]: Found {len(group_df)} rows at {group_key}")
    number_of_fragmented_points += 1

    # define a "template row" that we'll now dynamically change:
    merged_row = group_df.iloc[0].copy()

    for observable in observable_columnnames:

        values = group_df[observable].values
        values = values[~pd.isna(values)] # this finds the NaN values:

        if len(values) == 0:
            print(f"[WARN]: {group_key} (rows = {len(group_df)}) has NaNs")
            merged_row[observable] = 0.0 # replace with 0.0

        # find nonzero values:
        nonzero_values = values[~np.isclose(values, 0.0)]

        if len(nonzero_values) == 0:
            print(f"[WARN]: {group_key} (rows = {len(group_df)}) has all values are zero")
            number_of_all_zero_observables += 1
            merged_row[observable] = 0.0

        unique_nonzero_values = np.unique(nonzero_values)

        if len(unique_nonzero_values) == 1:

            merged_value = unique_nonzero_values[0]

            # print(f"[INFO]: Safe merge with value {merged_value}")
            merged_row[observable] = merged_value
            number_of_successful_merges += 1

        else:
            print(f"[ERROR]: {group_key} (rows = {len(group_df)}) has conflicting nonzero values detected")
            number_of_conflicts += 1
            # merged_row[observable] = unique_nonzero_values[0]
            merged_row[observable] = 0.0

    merged_observable_values = pd.to_numeric(merged_row[observable_columnnames], errors = 'coerce').values
    merged_observable_values = merged_observable_values[~pd.isna(merged_observable_values)]
    all_observables_zero = np.all(np.isclose(merged_observable_values, 0.0))

    if all_observables_zero:
        print(f"[WARN]: {group_key} (rows = {len(group_df)}) row has zeros for observables")
        number_of_fully_zero_rows_removed += 1
        continue

    merged_rows.append(merged_row)

df_removed_redunancies = pd.DataFrame(merged_rows)

print(f"[INFO]: Fragmented points found: {number_of_fragmented_points}")
print(f"[INFO]: Successful observable merges: {number_of_successful_merges}")
print(f"[INFO]: All-zero observable cases: {number_of_all_zero_observables}")
print(f"[INFO]: Conflicting observables detected: {number_of_conflicts}")
print(f"[INFO]: Final dataframe rows: {len(df_removed_redunancies)}")

### (3.15): Saving the refined dataframe!

In [ ]:
df_removed_redunancies.to_csv(
    path_or_buf = f"./local/version_{MAJOR_MINOR_NUMBER}/data/refined_experimental_data_v{MAJOR_MINOR_NUMBER}.csv",
    index = False)

print(f"[INFO]: Fragmented points: {number_of_fragmented_points}")
print(f"[INFO]: Successful merges: {number_of_successful_merges}")
print(f"[INFO]: All-zero kinematic settings: {number_of_all_zero_observables}")
print(f"[INFO]: Rows removed: {number_of_fully_zero_rows_removed}")
print(f"[INFO]: Conflicts: {number_of_conflicts}")

## (4): Making Plots:

### (4.1): Make a 3D Plot of the `gepard` Kinematic Coverage (Using Experimentally-Derived Pseudodata):

#### (4.1.1): Find the unique kinematic settings ($x_{\text{B}}, t, Q^{2}$) in the entire dataset:

In [ ]:
# [NOTE]: we drop duplicate of the subset "set" to avoid clogging up the 
# 3D-rendering. All we care about are *how many sets there are*, and what
# their kinematic points are:
unique_points = df_removed_redunancies.drop_duplicates(subset = ['set'])

#### (4.1.2): Actually make the plot:

In [ ]:
fig = plt.figure(figsize = (9, 7), layout = "tight")
ax = fig.add_subplot(1, 1, 1, projection = "3d")

ax.text2D(
    0.01, 0.00,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = ax.transAxes)

scatter = ax.scatter(
    unique_points['x_b'],
    unique_points['q_squared'],
    -1.0 * unique_points['t'],
    alpha = 0.3)

ax.set_title(rf"gepard Kinematic Coverage ($N_{{\textrm{{sets}}}} = {len(unique_points)}$)", fontsize = 18)
ax.set_xlabel(r"$x_{B}$", fontsize = 16.)
ax.set_ylabel(r"$Q^2$ [GeV$^2$]", fontsize = 16.)
ax.set_zlabel(r"$-t$ [GeV$^2$]", fontsize = 16.)

for extension in ['png', 'eps']:
    fig.savefig(
        f"./local/version_{MAJOR_MINOR_NUMBER}/plots/total_experimental_coverage_v{MAJOR_MINOR_NUMBER}.{extension}",
        facecolor = 'white', transparent = False)

plt.close(fig)

del fig
del ax

### (4.2): Scatterplots of Experimental Data:

#### (4.2.1): Determining the Unique Kinematic Settings for Plotting:

That means find the unique tuples ($x_{\text{B}}$, $t$, $Q^{2}$).

In [ ]:
# thanks Google Gemini!
kinematic_summary_dataframe = df_experimental_data.groupby(['experiment_year', 'set']).agg({
    'x_b': 'first', 't': 'first', 'q_squared': 'first'
}).reset_index()

print(f"[INFO]: Extracted {len(kinematic_summary_dataframe)} unique experiment-kinematic pairs.")
print(f"[INFO]: The number of unique kinematic points should be the same as the number we just found. Is it? {len(kinematic_summary_dataframe) == unique_sets_exp_data}")


What does this new DF look like?

In [ ]:
kinematic_summary_dataframe.head(7)

#### (4.2.2): Scatterplot of Experimental Datapoints *by Experiment!*

In [ ]:
all_uuxsec_experiments_figure = plt.figure(figsize = (9, 7), layout = "tight")
all_uuxsec_experiments_axis = all_uuxsec_experiments_figure.add_subplot(1, 1, 1, projection = "3d")

axis_elevation = all_uuxsec_experiments_axis.elev # extract eleveation param

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
all_uuxsec_experiments_axis.text2D(
    0.01, 0.00,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = all_uuxsec_experiments_axis.transAxes)

# for every experiment:
for experiment in kinematic_summary_dataframe['experiment_year'].unique():
    # quotient on the sub-dataframe that corresponds to that experiment:
    experiment_kinematic_set_data = kinematic_summary_dataframe[kinematic_summary_dataframe['experiment_year'] == experiment]

    # once quotiented, we can plot the kinematic quantities---which are also unique!
    all_uuxsec_experiments_axis.scatter(
        experiment_kinematic_set_data['x_b'],
        experiment_kinematic_set_data['q_squared'],
        -1.0 * experiment_kinematic_set_data['t'],
        label = experiment, alpha = 0.5)

all_uuxsec_experiments_axis.set_title(
    fr"gepard Experimental Coverage by Experiment ($N_{{\textrm{{sets}}}} = {len(unique_points)}$)", fontsize = 18)
all_uuxsec_experiments_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 16.)
all_uuxsec_experiments_axis.set_ylabel(r"$Q^{2}$ [GeV$^2$]", fontsize = 16.)
all_uuxsec_experiments_axis.set_zlabel(r"$-t$ [GeV$^2$]", fontsize = 16.)
all_uuxsec_experiments_axis.legend(fontsize = 12.)

for extension in ['png', 'eps']:
    all_uuxsec_experiments_figure.savefig(
        f"./local/version_{MAJOR_MINOR_NUMBER}/plots/experimental_kinematic_coverage_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)
    
plt.close(all_uuxsec_experiments_figure)

#### (4.2.3): 3D Scatterplot of All Experimental Datapoints *per Observable!*

This means that we make *two* scatterplots: one for cross-section and one for BSA.

In [ ]:
# define plot configuration items:
observable_plot_items = {
    "unp_beam_unp_target_xsec": {
        "title": r"$d^{4}\sigma^{UU}$",
        "file_suffix": "unp_beam_unp_target_xsec"
    },
    "unp_target_bsa": {
        "title": "BSA",
        "file_suffix": "unp_target_bsa"
    }
}

for column_name, plot_configuration in observable_plot_items.items():

    # quotient the dataframe on an *observable* instead of experiment:
    observable_dataframe = df_experimental_data[df_experimental_data[column_name] != 0].copy()
    
    experiment_name = plot_configuration['title']
    plot_observable = plot_configuration['file_suffix']

    if observable_dataframe.empty:
        print(f"[INFO]: No data found for {column_name}, skipping plot.")
        continue

    fig = plt.figure(figsize = (9, 7), layout = "tight")
    ax = fig.add_subplot(1, 1, 1, projection = "3d")

    ax.text2D(
        0.01, 0.00,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = ax.transAxes)

    # now quotient on the experiment:
    for experiment in observable_dataframe['experiment_year'].unique():

        # actually, this is where we quotient on the experiment:
        experiment_kinematic_set_data = observable_dataframe[observable_dataframe['experiment_year'] == experiment]

        # only obtain the kinematic point (to avoid high redundancy!)
        unique_kinematics = experiment_kinematic_set_data.drop_duplicates(subset = ['x_b', 'q_squared', 't'])
        
        ax.scatter(
            unique_kinematics['x_b'],
            unique_kinematics['q_squared'],
            -1.0 * unique_kinematics['t'],
            label = experiment, alpha = 0.6)

    ax.set_title(rf"Experimental Kinematic Settings Space for {experiment_name}", fontsize = 18)
    ax.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 16.)
    ax.set_ylabel(r"$Q^{2}$ [GeV$^2$]", fontsize = 16.)
    ax.set_zlabel(r"$-t$ [GeV$^2$]", fontsize = 16.)
    ax.legend(fontsize = 12.)

    # we have these limits to compare with the previous plots:
    ax.set_xlim(0.1, 0.6) # this is xb
    ax.set_ylim(1.0, 9.0) # this is q-squared
    ax.set_zlim(0.1, 1.5) # this is -t

    for extension in ['png', 'eps']:
        save_path = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/experimental_kinematic_space_for_{plot_observable}_v{MAJOR_MINOR_NUMBER}.{extension}"
        fig.savefig(
            save_path,
            facecolor = 'white', transparent = False,)
    
    print(f"[INFO]: Saved plot for {plot_observable}")
    plt.close(fig)

    # cleanup!
    del fig
    del ax

#### (4.2.4): Scatterplot of *Each Experimental Datapoints* Separately:

This block will plot *per individual experiment* the various kinematic settings that we probed *per observable!*

Firstly, what exactly are the experiments?

In [ ]:
kinematic_summary_dataframe['experiment_year'].unique()

In [ ]:
kinematic_summary_dataframe.columns

Excellent. We will quotient the dataset on these experiments.

But we actually need to do something first:

In [ ]:
# map all nonzero values of the observables to just a boolean flag:
coverage_map = df_experimental_data.groupby(['experiment_year', 'x_b', 'q_squared', 't']).agg({
    'unp_beam_unp_target_xsec': lambda x: (x != 0.0).any(),
    'unp_target_bsa': lambda x: (x != 0.0).any()
}).reset_index()

What does this look like?

In [ ]:
# a new DF with boolean flags for the presence of observable or not:
coverage_map.head(7)

Reassign the column names:

In [ ]:
# assign the columns names:
coverage_map.columns = ['experiment_year', 'x_b', 'q_squared', 't', 'has_xsec', 'has_bsa']

In [ ]:
# should show us new column names in the last two columns:
coverage_map.head(7)

In [ ]:
total_number_of_kinematic_points = 0

for experiment in coverage_map['experiment_year'].unique():
    
    single_experiment_figure = plt.figure(figsize = (9, 7), layout = "tight")
    single_experiment_axis = single_experiment_figure.add_subplot(1, 1, 1, projection = "3d")

    # quotient the dataset on the experiment year:
    experimental_subset = coverage_map[coverage_map['experiment_year'] == experiment]

    # these are DFs made with conditionals:
    both_observables = experimental_subset[experimental_subset['has_xsec'] & experimental_subset['has_bsa']]
    only_xsec = experimental_subset[experimental_subset['has_xsec'] & ~experimental_subset['has_bsa']]
    only_bsa = experimental_subset[~experimental_subset['has_xsec'] & experimental_subset['has_bsa']]
    neither = experimental_subset[~experimental_subset['has_xsec'] & ~experimental_subset['has_bsa']]

    # first entry is a DF:
    groups = [
        (only_xsec, 'red', r'$d^{{4}}\sigma^{{UU}}$'),
        (only_bsa, 'purple', r'$\textrm{BSA}$'),
        (both_observables, 'green', r'$d^{{4}}\sigma^{{UU}}$ and $\textrm{BSA}$'),
        (neither, 'grey', 'No Observable')
    ]

    # count the total number of datapoints:
    total_experimental_datapoints = 0
    
    # for every group:
    for dataframe, color, label in groups:

        if not dataframe.empty:
            print(f"[WARN]: Experiment {experiment} did measure {label}!")

            unique_kinematics = dataframe.drop_duplicates(subset = ['x_b', 'q_squared', 't'])
            # increment the total number of measured datapoints:
            total_experimental_datapoints += len(unique_kinematics)
            single_experiment_axis.scatter(
                unique_kinematics['x_b'], unique_kinematics['q_squared'], -1.0 * unique_kinematics['t'],
                label = label, color = color, alpha = 0.4
            )
            
        else:
            print(f"[WARN]: Experiment {experiment} did not measure {label}")
        
    total_number_of_kinematic_points += total_experimental_datapoints

    single_experiment_axis.text2D(
        -0.1, -0.1,
        fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = single_experiment_axis.transAxes)

    single_experiment_axis.set_title(rf"{experiment} Kinematic Coverage ($N_{{\textrm{{sets}}}} = {total_experimental_datapoints}$)", fontsize = 18)
    single_experiment_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 16)
    single_experiment_axis.set_ylabel(r"$Q^{2}$", fontsize = 16)
    single_experiment_axis.set_zlabel(r"$t$", fontsize = 16)
    single_experiment_axis.legend(fontsize = 16)

    # we have these limits to compare with the previous plots:
    single_experiment_axis.set_xlim(0.1, 0.6) # this is xb
    single_experiment_axis.set_ylim(1.0, 9.0) # this is q-squared
    single_experiment_axis.set_zlim(0.1, 1.5) # this is -t

    for extension in ['png', 'eps']:
        single_experiment_figure.savefig(
            f"./local/version_{MAJOR_MINOR_NUMBER}/plots/{experiment}_kinematic_space_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white', transparent = False)
    plt.close(single_experiment_figure)

    del single_experiment_figure
    del single_experiment_axis

print(f"[INFO]: Total number of experimental points probed: {total_number_of_kinematic_points}")

### (4.3): Actual Experimental Data: **[DANGER]:** Scatterplot of *Each Experimental Datapoints* Separately:
**[NOTE]: Generates a TON of plots --- be ready to run this!**

#### (4.3.1): Experimental Data for $d^{4}\sigma^{UU}$ **Only!**

In [ ]:
for kinematic_set_id, experiment in df_cross_section_no_issues.groupby('set'):
    exp_year = experiment['experiment_year'].iloc[0]
    coordinate_frame = experiment["coordinate_frame"].iloc[0]

    k_value, xb_value, t_value, q2_value = experiment[['k', 'x_b', 't', 'q_squared']].iloc[0]

    figure = plt.figure(figsize = (8, 7), layout = "tight")
    axis = figure.add_subplot(1, 1, 1)

    # Trento convention...
    x_data = experiment['phi'] if coordinate_frame == "BMK" else experiment['phi'] + np.pi

    axis.errorbar(
        x = x_data, y = experiment['unp_beam_unp_target_xsec'], yerr = experiment['unp_beam_unp_target_xsec_err'],
        fmt = 'o', capsize = 3, label = r'$d^{4}\sigma^{UU}$ [nb GeV$^{-4}$]'
    )
        
    title_string = fr"{exp_year} ({coordinate_frame}) $d^{{4}}\sigma^{{UU}}$ vs. $\phi$ (set {kinematic_set_id})"
    kinematics_string = (
        rf"$k = {k_value}$ GeV, "rf"$x_B = {xb_value}$, "
        rf"$t = {t_value}$ GeV$^2$, "rf"$Q^2 = {q2_value}$ GeV$^2$"
    )

    axis.set_title(f"{title_string}\n{kinematics_string}", fontsize = 16)
    axis.set_xlabel(r"$\phi$ (radians)", fontsize = 16)
    axis.set_ylabel(r"$d^{4}\sigma^{UU}$ [nb GeV$^{-4}$]", fontsize = 16)
    axis.grid(True, linestyle = '--', alpha = 0.6)

    axis.text(
        0.00, -0.05,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes,
        verticalalignment = 'top',
        horizontalalignment = 'left')
    
    for extension in ['png', 'eps']:
        figure.savefig(
            f"./local/version_{MAJOR_MINOR_NUMBER}/plots/setid_{kinematic_set_id}_{exp_year}_xsec_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)
    
    plt.close(figure)

    del figure
    del axis

#### (4.3.2): Experimental Data for $\text{BSA}$ **Only!**

In [ ]:
for kinematic_set_id, experiment in df_bsa_no_issues.groupby('set'):
    exp_year = experiment['experiment_year'].iloc[0]
    coordinate_frame = experiment["coordinate_frame"].iloc[0]

    k_value, xb_value, t_value, q2_value = experiment[['k', 'x_b', 't', 'q_squared']].iloc[0]

    figure = plt.figure(figsize = (8, 7), layout = "tight")
    axis = figure.add_subplot(1, 1, 1)

    x_data = experiment['phi'] if coordinate_frame == "BMK" else experiment['phi'] + np.pi

    axis.errorbar(
        x = x_data, y = experiment['unp_target_bsa'], yerr = experiment['unp_target_bsa_err'],
        fmt = 'o', capsize = 3, label = r'$\textrm{BSA}(\Lambda = 0)'
    )
        
    title_string = fr"{exp_year} ({coordinate_frame}) $\textrm{{BSA}}(\Lambda = 0)$ vs. $\phi$ (set {kinematic_set_id})"
    kinematics_string = (
        rf"$k = {k_value}$ GeV, "rf"$x_B = {xb_value}$, "
        rf"$t = {t_value}$ GeV$^2$, "rf"$Q^2 = {q2_value}$ GeV$^2$"
    )

    axis.set_title(f"{title_string}\n{kinematics_string}", fontsize = 16)
    axis.set_xlabel(r"$\phi$ (radians)", fontsize = 16)
    axis.set_ylabel(r"$\textrm{BSA}(\Lambda = 0)$", fontsize = 16)
    axis.grid(True, linestyle = '--', alpha = 0.6)

    axis.text(
        0.00, -0.05,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes,
        verticalalignment = 'top',
        horizontalalignment = 'left'
    )
    
    for extension in ['png', 'eps']:
        figure.savefig(
            f"./local/version_{MAJOR_MINOR_NUMBER}/plots/setid_{kinematic_set_id}_{exp_year}_bsa_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)
    
    plt.close(figure)

    del figure
    del axis

### (3.6): Making BSA and Cross-Section Data:

In [ ]:
df = pd.read_csv(
    f"./local/version_{MAJOR_MINOR_NUMBER}/data/refined_experimental_data_v{MAJOR_MINOR_NUMBER}.csv")
    
observables = [
    'unp_beam_unp_target_xsec', 
    'unp_target_bsa', 
]

nonzero_counts = (df[observables] != 0).sum(axis = 1)

df_filtered = df[nonzero_counts >= 2].copy()

dropped_mask = nonzero_counts < 2
dropped_rows = df[dropped_mask]

dropped_sets = dropped_rows["set"].value_counts().sort_index()

print("Dropped set counts:")
print(dropped_sets)

# Optional: also keep a full log dataframe
dropped_log = dropped_rows[["set"] + observables]

df_filtered.to_csv(f"./local/version_{MAJOR_MINOR_NUMBER}/data/combined_xsec_bsa_experimental_data_v{MAJOR_MINOR_NUMBER}.csv", index = False)

### (4.2): Partition the Huge Datafile into Individual Kinematic Sets

**[WARNING]: Makes a TON of folders --- be ready to run this!**

In [ ]:
main_experimental_datafile = pd.read_csv(
    f"./local/version_{MAJOR_MINOR_NUMBER}/data/experimental_data_v{MAJOR_MINOR_NUMBER}.csv")

for kinematic_set_number, kinematic_group in main_experimental_datafile.groupby('set'):

    os.makedirs(f"./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/data", exist_ok = True)
    os.makedirs(f"./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/plots", exist_ok = True)
    os.makedirs(f"./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/learning_curves", exist_ok = True)
    os.makedirs(f"./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/replicas", exist_ok = True)
    
    set_data = main_experimental_datafile[main_experimental_datafile['set'] == kinematic_set_number]

    set_data.to_csv(
        f"./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/data/main_experimental_file_v{MAJOR_MINOR_NUMBER}.csv", 
        index = False)
    
    # if it's the same kinematic setting, then these mean()s should be the equivalence class value...
    fixed_k = kinematic_group['k'].mean()
    fixed_q_squared = kinematic_group['q_squared'].mean()
    fixed_x_bjorken = kinematic_group['x_b'].mean()
    fixed_t = kinematic_group['t'].mean()
    number_of_phi_points = kinematic_group['phi'].nunique()
    
    print(f"[INFO]: Processed Set {kinematic_set_number}!")

    with open(
        file = f"./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/log_set_{kinematic_set_number}_v{MAJOR_MINOR_NUMBER}.txt",
        mode = "w",
        encoding = "utf-8",
        buffering = 1) as logfile:

        logfile.write(f"[INFO]: Kinematic Set Number: {kinematic_set_number}\n")
        logfile.write(f"[INFO]: k = {fixed_k}\n")
        logfile.write(f"[INFO]: Q^2 = {fixed_q_squared}\n")
        logfile.write(f"[INFO]: xb = {fixed_x_bjorken}\n")
        logfile.write(f"[INFO]: t = {fixed_t}\n")

        fixed_ep = compute_epsilon(fixed_x_bjorken, fixed_q_squared)
        fixed_y = compute_y(fixed_k, fixed_q_squared, fixed_ep)
        fixed_xi = compute_skewness(fixed_x_bjorken, fixed_t, fixed_q_squared)
        fixed_t_min = compute_t_min(fixed_x_bjorken, fixed_q_squared, fixed_ep)
        fixed_k_tilde = compute_k_tilde(fixed_x_bjorken, fixed_q_squared, fixed_t, fixed_t_min, fixed_ep)
        fixed_big_k = compute_k(fixed_q_squared, fixed_y, fixed_ep, fixed_k_tilde)
        fixed_t_prime = compute_t_prime(fixed_t, fixed_t_min)
        fixed_fe = compute_fe(fixed_t)
        fixed_fg = compute_fg(fixed_fe)
        fixed_f2 = compute_f2(fixed_t, fixed_fe, fixed_fg)
        fixed_f1 = compute_f1(fixed_fg, fixed_f2)

        logfile.write(f"[INFO]: ep = {fixed_ep}\n")
        logfile.write(f"[INFO]: y = {fixed_y}\n")
        logfile.write(f"[INFO]: xi = {fixed_xi}\n")
        logfile.write(f"[INFO]: tmin = {fixed_t_min}\n")
        logfile.write(f"[INFO]: Ktilde = {fixed_k_tilde}\n")
        logfile.write(f"[INFO]: K = {fixed_big_k}\n")
        logfile.write(f"[INFO]: tprime = {fixed_t_prime}\n")
        logfile.write(f"[INFO]: FE = {fixed_fe}\n")
        logfile.write(f"[INFO]: FG = {fixed_fg}\n")
        logfile.write(f"[INFO]: F2 = {fixed_f2}\n")
        logfile.write(f"[INFO]: F1 = {fixed_f1}\n")

        logfile.write(f"[INFO]: Made new directory at path: ./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/data\n")
        logfile.write(f"[INFO]: Made new directory at path: ./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/plots\n")
        logfile.write(f"[INFO]: Made new directory at path: ./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/replicas\n")
        logfile.write(f"[INFO]: Made new directory at path: ./local/version_{MAJOR_MINOR_NUMBER}/kinematic_set_{kinematic_set_number}/learning_curves\n")

        logfile.write(f"[INFO]: Each point has {number_of_phi_points} phi values\n")
        logfile.close()

print("[INFO]: End of script reached!")
